In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

FILE=r"C:\Users\YASH\Desktop\BrainyBeam\task3 web\Water Quality.csv"
LOCATION="MonitoringLocationID"
PARAM="CharacteristicName"
VALUE="ResultValue"
DATE="ActivityStartDate"
AREA_COLUMN="AreaType"  # Change only if your CSV has a real urban/rural label
OUT="task2_clustering_outputs"
os.makedirs(OUT, exist_ok=True)
df=pd.read_csv(FILE, low_memory=False)
required=[LOCATION,PARAM,VALUE]
missing=[c for c in required if c not in df.columns]
if missing: raise ValueError(f"Missing columns: {missing}; available: {list(df.columns)}")
df[VALUE]=pd.to_numeric(df[VALUE],errors="coerce")
df=df.dropna(subset=required).copy()
keys=[LOCATION]
if DATE in df.columns:
    dates=pd.to_datetime(df[DATE],errors="coerce")
    if dates.notna().any():
        df[DATE]=dates.dt.to_period("Y").astype(str)
        keys.append(DATE)
wide=df.pivot_table(index=keys,columns=PARAM,values=VALUE,aggfunc="median")
wide=wide.dropna(axis=1,thresh=max(3,int(len(wide)*.20)))
wide=wide.loc[:,wide.nunique(dropna=True)>1]
if len(wide)<3 or wide.shape[1]<2:
    raise ValueError("Need at least 3 samples and 2 varying parameters. Check missing data or reduce feature-coverage threshold.")
X=SimpleImputer(strategy="median").fit_transform(wide)
X=StandardScaler().fit_transform(X)
search=[]; best=None
for k in range(2,min(7,len(wide))):
    model=KMeans(n_clusters=k,random_state=42,n_init=10)
    labels=model.fit_predict(X)
    if len(np.unique(labels))<2 or len(np.unique(labels))>=len(labels): continue
    score=silhouette_score(X,labels)
    search.append({"k":k,"silhouette_score":score,"inertia":model.inertia_})
    if best is None or score>best["score"]: best={"model":model,"labels":labels,"score":score,"k":k}
if best is None: raise ValueError("Could not create valid K-Means clusters for silhouette scoring.")
pd.DataFrame(search).to_csv(f"{OUT}/kmeans_parameter_search.csv",index=False)
print("K-Means parameter search:\n",pd.DataFrame(search).round(4).to_string(index=False))
print(f"Selected k={best['k']}; silhouette={best['score']:.4f}")
db=[]
for eps in [.5,1,1.5,2]:
    labels=DBSCAN(eps=eps,min_samples=5).fit_predict(X)
    keep=labels!=-1; clusters=set(labels[keep]); score=np.nan
    if len(clusters)>=2 and keep.sum()>len(clusters):
        try: score=silhouette_score(X[keep],labels[keep])
        except Exception: pass
    db.append({"eps":eps,"min_samples":5,"clusters_excluding_noise":len(clusters),
               "noise_points":int((labels==-1).sum()),"silhouette_excluding_noise":score})
pd.DataFrame(db).to_csv(f"{OUT}/dbscan_parameter_search.csv",index=False)
result=wide.copy(); result["Cluster"]=best["labels"]
result.to_csv(f"{OUT}/cluster_assignments.csv")
result.groupby("Cluster")[list(wide.columns)].mean().to_csv(f"{OUT}/cluster_feature_means.csv")
print("Cluster sizes:\n",result["Cluster"].value_counts().sort_index())
if AREA_COLUMN not in df.columns:
    print(f"No '{AREA_COLUMN}' label found. These are water-quality pattern clusters, not verified urban/rural groups.")
coords=PCA(n_components=2).fit_transform(X)
plt.figure(figsize=(8,6))
for cl in sorted(np.unique(best["labels"])):
    mask=best["labels"]==cl
    plt.scatter(coords[mask,0],coords[mask,1],alpha=.7,s=22,label=f"Cluster {cl}")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("Water-quality clusters projected onto PCA")
plt.legend(); plt.tight_layout(); plt.savefig(f"{OUT}/cluster_pca_plot.png",dpi=220); plt.close()
print(f"Done. Outputs: {OUT}/")
print("Silhouette ranges from -1 to 1; higher is generally better-separated. Cluster IDs have no inherent meaning.")

K-Means parameter search:
  k  silhouette_score    inertia
 2            0.5230 45617.1428
 3            0.4511 41782.9608
 4            0.4590 39222.0079
 5            0.3571 37051.0025
 6            0.1520 35590.6464
Selected k=2; silhouette=0.5230
Cluster sizes:
 Cluster
0    1047
1     148
Name: count, dtype: int64
No 'AreaType' label found. These are water-quality pattern clusters, not verified urban/rural groups.
Done. Outputs: task2_clustering_outputs/
Silhouette ranges from -1 to 1; higher is generally better-separated. Cluster IDs have no inherent meaning.
